# Run Experiment G on Colab

This notebook loads the final step-171 LoRA on the text backbone of Qwen3.5-4B. It downloads the exact public base revision used in the experiment and loads the adapter and tokenizer from `experiment_g_colab_adapter.zip`.

In Colab, select **Runtime → Change runtime type → GPU**. An L4 is a good choice if available. Run the cells from top to bottom, choose the adapter ZIP when prompted, and edit the question in the last cell. The base model download takes several GB. The first generation uses a smaller token budget than the experiment for interactive use.

The notebook uses Colab's PyTorch build. The base model is public and inference uses no judge service or experiment credentials.

The last cell streams the base response followed by the trained response. Both use the same prompt, sampling settings, and seed. Turn on `stream_token_ids` to print one line per generated token with its ID and raw vocabulary piece.


In [ ]:
# Install the inference libraries used in the run.
import subprocess
import sys

# This notebook loads regular FP16 or BF16 weights. Remove the optional
# torchao package so an older Colab build cannot block PEFT adapter loading.
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.20.2", "peft==0.21.2", "accelerate==1.15.0",
])


In [ ]:
# Choose the adapter ZIP, or set a path to a file already in this runtime.
import hashlib
from io import BytesIO
import json
from pathlib import Path
from zipfile import ZipFile

zip_path = ""  # Example after uploading in the Files sidebar: /content/experiment_g_colab_adapter.zip
if zip_path:
    archive_name = zip_path
    archive_stream = Path(zip_path).open("rb")
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose exactly one file, experiment_g_colab_adapter.zip.")
    archive_name, contents = next(iter(uploaded.items()))
    archive_stream = BytesIO(contents)
    del uploaded, contents

bundle_dir = Path("/content/experiment_g")
adapter_dir = bundle_dir / "adapter"
expected_files = (
    "adapter/adapter_config.json", "adapter/adapter_model.safetensors",
    "adapter/chat_template.jinja", "adapter/tokenizer.json",
    "adapter/tokenizer_config.json", "prompts.json",
)
with archive_stream as stream:
    stream.seek(0, 2)
    size = stream.tell()
    stream.seek(0)
    print(f"Received {archive_name!r} ({size:,} bytes).")
    expected_sha256 = "ea1092f43b203fd0217f0008faf3767d3061222dcac0aa4b87664cbdc2a77acc"
    if hashlib.file_digest(stream, "sha256").hexdigest() != expected_sha256:
        raise ValueError(
            "This file does not match experiment_g_colab_adapter.zip (124,937,418 bytes). "
            "Use the original ZIP from the Experiment G output folder. "
            "If you already selected it, upload it again and wait for the transfer to finish."
        )
    stream.seek(0)
    with ZipFile(stream) as bundle:
        for name in expected_files:
            bundle.extract(name, bundle_dir)

with (adapter_dir / "adapter_model.safetensors").open("rb") as handle:
    checksum = hashlib.file_digest(handle, "sha256").hexdigest()
if checksum != "5960eed8b5c0028ba6bc4a507f89858d746631b1df77c6f600e5ffe87509f61d":
    raise ValueError("The adapter checksum does not match the final Experiment G checkpoint.")

prompt_registry = json.loads((bundle_dir / "prompts.json").read_text())
adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text())
print("Verified the Experiment G adapter.")


In [ ]:
# Load the text model with the same weight mapping as Experiment G.
import torch
from huggingface_hub import hf_hub_download
from peft import PeftModel
from transformers import AutoTokenizer, Qwen3_5ForCausalLM, Qwen3_5TextConfig

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then rerun the notebook.")

base_id = "Qwen/Qwen3.5-4B"
revision = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
if (adapter_config["base_model_name_or_path"], adapter_config["revision"]) != (base_id, revision):
    raise ValueError("The adapter was saved for a different base model or revision.")

dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
raw_config = json.loads(Path(hf_hub_download(base_id, "config.json", revision=revision)).read_text())
text_config = Qwen3_5TextConfig.from_dict(raw_config["text_config"])
text_config.architectures = ["Qwen3_5ForCausalLM"]
text_config.tie_word_embeddings = bool(
    raw_config.get("tie_word_embeddings", raw_config["text_config"].get("tie_word_embeddings", False))
)

base, loading = Qwen3_5ForCausalLM.from_pretrained(
    base_id,
    revision=revision,
    config=text_config,
    key_mapping={r"^model\.language_model\.": "model."},
    output_loading_info=True,
    dtype=dtype,
    device_map={"": 0},
    attn_implementation="sdpa",
)
if any(loading.get(key) for key in ("missing_keys", "mismatched_keys", "error_msgs")):
    raise RuntimeError(f"Some base model weights could not be loaded: {loading}")

tokenizer = AutoTokenizer.from_pretrained(adapter_dir, local_files_only=True)
model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
model.eval()
print(f"Model loaded on {torch.cuda.get_device_name(0)} with {dtype}.")


In [ ]:
# Edit this cell and rerun it to ask another question.
from transformers import LogitsProcessor, LogitsProcessorList, TextStreamer, set_seed
from transformers.generation.streamers import BaseStreamer

question = "Why does the singular value decomposition exist even for a defective matrix?"  # @param {type:"string"}
subject = "math"  # @param ["science", "physics", "chemistry", "biology", "math", "earth_science", "geography", "medicine"]
max_new_tokens = 2048  # @param {type:"integer"}
seed = 123  # @param {type:"integer"}
stream_token_ids = False  # @param {type:"boolean"}

system_prompt = prompt_registry["resolved_prompts"][subject]
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question},
]
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, return_dict=False, add_generation_prompt=True, enable_thinking=True,
)
input_ids = torch.tensor([prompt_ids], device="cuda:0")
prompt_length = input_ids.shape[1]

class PresencePenalty(LogitsProcessor):
    def __call__(self, input_ids, scores):
        for index, row in enumerate(input_ids):
            seen = row[prompt_length:].unique()
            scores[index, seen] -= 1.5
        return scores

eos = model.generation_config.eos_token_id
eos_ids = [eos] if isinstance(eos, int) else list(eos or [])
if tokenizer.eos_token_id not in eos_ids:
    eos_ids.append(tokenizer.eos_token_id)
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

class TokenIdStreamer(BaseStreamer):
    """Print the ID and vocabulary piece of each generated token."""

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.next_is_prompt = True

    def put(self, value):
        if self.next_is_prompt:
            self.next_is_prompt = False
            return
        for token_id in value.reshape(-1).tolist():
            piece = self.tokenizer.convert_ids_to_tokens(token_id)
            print(f"{token_id:>6}  {piece!r}", flush=True)

    def end(self):
        self.next_is_prompt = True

def stream_response(label):
    print(f"\n=== {label} ===", flush=True)
    if stream_token_ids:
        streamer = TokenIdStreamer(tokenizer)
    else:
        streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=False,
                                clean_up_tokenization_spaces=False)

    set_seed(seed)
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            top_k=20,
            min_p=0.0,
            repetition_penalty=1.0,
            logits_processor=LogitsProcessorList([PresencePenalty()]),
            eos_token_id=eos_ids,
            pad_token_id=pad_id,
            use_cache=True,
            streamer=streamer,
        )

    generated = output[0, prompt_length:].tolist()
    closing_token = tokenizer.convert_tokens_to_ids("</think>")
    ended_with_eos = bool(generated) and generated[-1] in eos_ids
    if ended_with_eos and closing_token not in generated:
        print("\nStopped at an end token before closing the thinking trace. "
              "This sample has no separate final answer. Try a different seed.")
    elif len(generated) >= max_new_tokens and not ended_with_eos:
        print("\nGeneration reached the token limit. Increase max_new_tokens for a longer answer.")

with model.disable_adapter():
    stream_response("Base Qwen3.5-4B")

stream_response("Experiment G trained adapter")
